<a href="https://colab.research.google.com/github/satvik959/DL-2025-26/blob/main/exporing_ciphar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print("Classes:", class_names)


class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)


def train_model(model, optimizer, criterion, train_loader, epochs=5):
    model = model.to(device)
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}  Train Accuracy: {epoch_acc:.2f}%")

    total_time = time.time() - start_time
    return total_time


def evaluate_model(model, test_loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    return acc, prec, rec, f1, cm, y_true, y_pred


def run_experiment(optimizer_name, optimizer):
    print("\n" + "=" * 60)
    print("Running with optimizer:", optimizer_name)
    print("=" * 60)

    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    total_time = train_model(model, optimizer(model.parameters()), criterion, train_loader, epochs=5)

    acc, prec, rec, f1, cm, y_true, y_pred = evaluate_model(model, test_loader)

    print("\nTest Results for", optimizer_name)
    print(f"Accuracy : {acc * 100:.2f}%")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"Time     : {total_time:.2f} seconds")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    return {
        "optimizer": optimizer_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "time": total_time,
        "cm": cm
    }


results = []

results.append(run_experiment("SGD", lambda params: optim.SGD(params, lr=0.01)))
results.append(run_experiment("SGD + Momentum", lambda params: optim.SGD(params, lr=0.01, momentum=0.9)))
results.append(run_experiment("RMSProp", lambda params: optim.RMSprop(params, lr=0.001)))
results.append(run_experiment("Adam", lambda params: optim.Adam(params, lr=0.001)))
results.append(run_experiment("AdamW", lambda params: optim.AdamW(params, lr=0.001)))


print("\n" + "=" * 80)
print("FINAL COMPARISON")
print("=" * 80)

for r in results:
    print(f"{r['optimizer']:<15} -> Accuracy: {r['accuracy']*100:.2f}% | Precision: {r['precision']:.4f} | Recall: {r['recall']:.4f} | F1: {r['f1']:.4f} | Time: {r['time']:.2f}s")

Using device: cpu


100%|██████████| 170M/170M [00:02<00:00, 62.8MB/s]


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Running with optimizer: SGD
Epoch [1/5]  Loss: 2.2646  Train Accuracy: 20.26%
Epoch [2/5]  Loss: 2.0969  Train Accuracy: 25.36%
Epoch [3/5]  Loss: 1.9375  Train Accuracy: 30.82%
Epoch [4/5]  Loss: 1.8352  Train Accuracy: 34.62%
Epoch [5/5]  Loss: 1.7552  Train Accuracy: 37.53%

Test Results for SGD
Accuracy : 39.22%
Precision: 0.3850
Recall   : 0.3922
F1-score : 0.3784
Time     : 168.27 seconds

Classification Report:
              precision    recall  f1-score   support

    airplane       0.47      0.46      0.47      1000
  automobile       0.44      0.47      0.45      1000
        bird       0.35      0.12      0.18      1000
         cat       0.27      0.15      0.19      1000
        deer       0.35      0.35      0.35      1000
         dog       0.37      0.37      0.37      1000
        frog       0.35      0.55      0.43      1000
       horse       0.36      0.41      0.38 

MLP Observation:
The CIFAR-10 dataset is more suitable for CNN-based models, as CNNs are designed to preserve and learn spatial features from images. The MLP model shows lower accuracy because it converts the image into a flat vector, causing the loss of important spatial relationships between pixels. In addition, the absence of regularization methods such as Dropout, L2 regularization, and Batch Normalization further affects the model’s ability to generalize well. Therefore, even with a sufficient number of epochs, the accuracy remains lower compared to CNN models.

In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])


full_train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

val_dataset.dataset.transform = test_transform

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

class_names = test_dataset.classes
print("Classes:", class_names)


class RegularizedMLP(nn.Module):
    def __init__(self):
        super(RegularizedMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(32 * 32 * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    return avg_loss, acc, prec, rec, f1, cm, y_true, y_pred


def train_model(model, train_loader, val_loader, epochs=20, patience=5):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    patience_counter = 0

    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = 100 * correct / total

        val_loss, val_acc, val_prec, val_rec, val_f1, _, _, _ = evaluate_model(model, val_loader, criterion)

        scheduler.step(val_loss)

        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc*100:.2f}%")
        print("-" * 50)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    total_time = time.time() - start_time
    model.load_state_dict(best_model_wts)

    return model, total_time, criterion


model = RegularizedMLP()
model, total_time, criterion = train_model(model, train_loader, val_loader, epochs=5, patience=5)


test_loss, test_acc, test_prec, test_rec, test_f1, cm, y_true, y_pred = evaluate_model(model, test_loader, criterion)

print("\nFINAL TEST RESULTS")
print(f"Test Loss : {test_loss:.4f}")
print(f"Accuracy  : {test_acc * 100:.2f}%")
print(f"Precision : {test_prec:.4f}")
print(f"Recall    : {test_rec:.4f}")
print(f"F1-score  : {test_f1:.4f}")
print(f"Time      : {total_time:.2f} seconds")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

print("\nConfusion Matrix:")
print(cm)

Using device: cpu
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Epoch [1/5]
Train Loss: 1.8732 | Train Acc: 36.77%
Val   Loss: 1.6946 | Val   Acc: 44.91%
--------------------------------------------------
Epoch [2/5]
Train Loss: 1.7220 | Train Acc: 43.86%
Val   Loss: 1.6254 | Val   Acc: 48.14%
--------------------------------------------------
Epoch [3/5]
Train Loss: 1.6568 | Train Acc: 46.95%
Val   Loss: 1.5836 | Val   Acc: 50.48%
--------------------------------------------------
Epoch [4/5]
Train Loss: 1.6142 | Train Acc: 49.30%
Val   Loss: 1.5671 | Val   Acc: 51.41%
--------------------------------------------------
Epoch [5/5]
Train Loss: 1.5813 | Train Acc: 50.60%
Val   Loss: 1.5347 | Val   Acc: 52.68%
--------------------------------------------------

FINAL TEST RESULTS
Test Loss : 1.5326
Accuracy  : 52.47%
Precision : 0.5230
Recall    : 0.5247
F1-score  : 0.5212
Time      : 204.22 seconds

Classification Report:
           

The results may remain similar because MLP itself is not well-suited for CIFAR-10, as it loses spatial information by flattening the image. Also, in the current code, both training and validation subsets share the same dataset object, so changing the validation transform may unintentionally remove augmentation from the training set as well. In addition, regularization methods usually improve generalization gradually and may not always cause a large jump in final accuracy, especially on a difficult image dataset with an MLP.

In [1]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------------------------------
# 1. Data
# -------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_dataset_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
val_dataset_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=test_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, _ = random_split(train_dataset_full, [train_size, val_size])
_, val_dataset = random_split(val_dataset_full, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

# -------------------------------------------------
# 2. MLP Model
# -------------------------------------------------
class RegularizedMLP(nn.Module):
    def __init__(self, dropout1=0.4, dropout2=0.4, dropout3=0.3):
        super(RegularizedMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(32 * 32 * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout1),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout3),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)

# -------------------------------------------------
# 3. Evaluation
# -------------------------------------------------
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return avg_loss, acc, prec, rec, f1

# -------------------------------------------------
# 4. Training
# -------------------------------------------------
def train_model(model, train_loader, val_loader, lr=0.001, weight_decay=1e-4, epochs=10, patience=3):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = 100 * correct / total

        val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate_model(model, val_loader, criterion)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc*100:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    total_time = time.time() - start_time
    model.load_state_dict(best_model_wts)

    test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate_model(model, test_loader, criterion)

    return {
        "accuracy": test_acc,
        "precision": test_prec,
        "recall": test_rec,
        "f1": test_f1,
        "time": total_time
    }

experiments = [
    {"name": "Baseline", "lr": 0.001, "drop1": 0.4, "drop2": 0.4, "drop3": 0.3, "wd": 1e-4},
    {"name": "Low Dropout", "lr": 0.001, "drop1": 0.2, "drop2": 0.2, "drop3": 0.2, "wd": 1e-4},
    {"name": "High Dropout", "lr": 0.001, "drop1": 0.5, "drop2": 0.5, "drop3": 0.4, "wd": 1e-4},
    {"name": "Low LR", "lr": 0.0005, "drop1": 0.4, "drop2": 0.4, "drop3": 0.3, "wd": 1e-4},
    {"name": "High LR", "lr": 0.005, "drop1": 0.4, "drop2": 0.4, "drop3": 0.3, "wd": 1e-4},
    {"name": "No L2", "lr": 0.001, "drop1": 0.4, "drop2": 0.4, "drop3": 0.3, "wd": 0},
    {"name": "High L2", "lr": 0.001, "drop1": 0.4, "drop2": 0.4, "drop3": 0.3, "wd": 1e-3},
]

results = []

for exp in experiments:
    print("\n" + "=" * 70)
    print("Running Experiment:", exp["name"])
    print("=" * 70)

    model = RegularizedMLP(
        dropout1=exp["drop1"],
        dropout2=exp["drop2"],
        dropout3=exp["drop3"]
    )

    result = train_model(
        model,
        train_loader,
        val_loader,
        lr=exp["lr"],
        weight_decay=exp["wd"],
        epochs=10,
        patience=3
    )

    result["name"] = exp["name"]
    results.append(result)

print("\n" + "=" * 90)
print("FINAL COMPARISON")
print("=" * 90)

for r in results:
    print(f"{r['name']:<15} -> Accuracy: {r['accuracy']*100:.2f}% | Precision: {r['precision']:.4f} | Recall: {r['recall']:.4f} | F1: {r['f1']:.4f} | Time: {r['time']:.2f}s")

Using device: cuda


100%|██████████| 170M/170M [00:14<00:00, 12.0MB/s]



Running Experiment: Baseline
Epoch [1/10] | Train Loss: 1.9912 | Train Acc: 30.18% | Val Acc: 39.81%
Epoch [2/10] | Train Loss: 1.8616 | Train Acc: 36.72% | Val Acc: 42.65%
Epoch [3/10] | Train Loss: 1.8157 | Train Acc: 38.84% | Val Acc: 45.45%
Epoch [4/10] | Train Loss: 1.7872 | Train Acc: 40.39% | Val Acc: 46.61%
Epoch [5/10] | Train Loss: 1.7598 | Train Acc: 41.76% | Val Acc: 47.33%
Epoch [6/10] | Train Loss: 1.7431 | Train Acc: 42.44% | Val Acc: 47.70%
Epoch [7/10] | Train Loss: 1.7294 | Train Acc: 43.22% | Val Acc: 48.37%
Epoch [8/10] | Train Loss: 1.7185 | Train Acc: 43.63% | Val Acc: 49.47%
Epoch [9/10] | Train Loss: 1.7071 | Train Acc: 44.39% | Val Acc: 50.07%
Epoch [10/10] | Train Loss: 1.6956 | Train Acc: 44.56% | Val Acc: 50.46%

Running Experiment: Low Dropout
Epoch [1/10] | Train Loss: 1.9438 | Train Acc: 32.74% | Val Acc: 39.71%
Epoch [2/10] | Train Loss: 1.8159 | Train Acc: 39.01% | Val Acc: 44.13%
Epoch [3/10] | Train Loss: 1.7602 | Train Acc: 41.74% | Val Acc: 46.28%


First the epochs has been increased to 10 and low drop out has given a better output than other ones as the neurons removed is less rather than other ones

In [2]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_dataset_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
val_dataset_full = datasets.CIFAR10(root="./data", train=True, download=False, transform=test_transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, _ = random_split(train_dataset_full, [train_size, val_size])
_, val_dataset = random_split(val_dataset_full, [train_size, val_size])
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

class RegularizedMLP(nn.Module):
    def __init__(self):
        super(RegularizedMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(32 * 32 * 3, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    return avg_loss, acc, prec, rec, f1

def train_model(train_loader, val_loader, batch_size, epochs=10, patience=3):
    model = RegularizedMLP().to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    patience_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = 100 * correct / total

        val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate_model(model, val_loader, criterion)

        print(f"Batch Size {batch_size} | Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc*100:.2f}%")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    total_time = time.time() - start_time
    model.load_state_dict(best_model_wts)

    test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate_model(model, test_loader, criterion)

    return {
        "batch_size": batch_size,
        "accuracy": test_acc,
        "precision": test_prec,
        "recall": test_rec,
        "f1": test_f1,
        "time": total_time
    }

batch_sizes = [32, 64, 128, 256]
results = []

for bs in batch_sizes:
    print("\n" + "=" * 70)
    print("Running experiment for batch size:", bs)
    print("=" * 70)

    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=bs, shuffle=False, num_workers=2)

    result = train_model(train_loader, val_loader, batch_size=bs, epochs=10, patience=3)
    results.append(result)

print("\n" + "=" * 90)
print("FINAL BATCH SIZE COMPARISON")
print("=" * 90)

for r in results:
    print(f"Batch Size {r['batch_size']:<4} -> Accuracy: {r['accuracy']*100:.2f}% | Precision: {r['precision']:.4f} | Recall: {r['recall']:.4f} | F1: {r['f1']:.4f} | Time: {r['time']:.2f}s")

Using device: cuda

Running experiment for batch size: 32
Batch Size 32 | Epoch [1/10] | Train Loss: 1.9706 | Train Acc: 31.18% | Val Acc: 40.30%
Batch Size 32 | Epoch [2/10] | Train Loss: 1.8485 | Train Acc: 37.58% | Val Acc: 45.67%
Batch Size 32 | Epoch [3/10] | Train Loss: 1.7998 | Train Acc: 39.87% | Val Acc: 46.34%
Batch Size 32 | Epoch [4/10] | Train Loss: 1.7647 | Train Acc: 41.79% | Val Acc: 47.99%
Batch Size 32 | Epoch [5/10] | Train Loss: 1.7425 | Train Acc: 42.77% | Val Acc: 48.90%
Batch Size 32 | Epoch [6/10] | Train Loss: 1.7273 | Train Acc: 43.19% | Val Acc: 50.74%
Batch Size 32 | Epoch [7/10] | Train Loss: 1.7084 | Train Acc: 44.26% | Val Acc: 51.67%
Batch Size 32 | Epoch [8/10] | Train Loss: 1.6947 | Train Acc: 45.14% | Val Acc: 52.06%
Batch Size 32 | Epoch [9/10] | Train Loss: 1.6785 | Train Acc: 45.64% | Val Acc: 52.84%
Batch Size 32 | Epoch [10/10] | Train Loss: 1.6656 | Train Acc: 46.73% | Val Acc: 53.32%

Running experiment for batch size: 64
Batch Size 64 | Epoch 

as the batch size increases the model can look at more images at once and then calculate loss accordingly and compute the accuracy params .